# ラボ 3b: BigQuery ML モデルの線形特徴量エンジニアリング / Transform

**学習目標**

1. BigQuery の `ML.FEATURE_CROSS` を使用した線形モデルの作成と評価
1. BigQuery の `ML.FEATURE_CROSS` および `ML.BUCKETIZE` を使用した線形モデルの作成と評価
1. `ML.TRANSFORM` を使用した線形モデルの作成と評価


## はじめに
このノートブックでは、BigQuery ML を使用して段階的に高度な特徴量エンジニアリングを適用しながら、出産前の赤ちゃんの体重を予測する複数の線形モデルを作成します。復習が必要な場合は、前のノートブック [BQML ベースライン モデル](../solutions/3a_bqml_baseline_babyweight.ipynb) に戻ってベースライン モデルの作成方法を確認できます。

BigQuery の `ML.FEATURE_CROSS` を使用した線形モデルの作成と評価、BigQuery の `ML.FEATURE_CROSS` および `ML.BUCKETIZE` を使用した線形モデルの作成と評価、そして BigQuery の `ML.TRANSFORM` を使用した線形モデルの作成と評価を行います。

各学習目標は受講者用ラボ ノートブック内の __#TODO__ に対応しています。まずこのノートブックを完了してから、[解答ノートブック](../solutions/3b_bqml_linear_transform_babyweight.ipynb) を確認してください。

## テーブルが存在することの確認

次のセルを実行して、以前のラボでデータセットとデータ テーブルが作成されていることを確認します。作成されていない場合は、ラボ [1b_prepare_data_babyweight](../solutions/1b_prepare_data_babyweight.ipynb) に戻って作成してください。

In [ ]:
%load_ext google.cloud.bigquery

In [ ]:
%%bigquery
-- LIMIT 0 は無料のクエリです。これによりテーブルが存在することを確認できます。
SELECT * FROM babyweight.babyweight_data_train
LIMIT 0

In [ ]:
%%bigquery
-- LIMIT 0 は無料のクエリです。これによりテーブルが存在することを確認できます。
SELECT * FROM babyweight.babyweight_data_eval
LIMIT 0

## ラボ課題 1: モデル 1: カテゴリカル特徴量への ML.FEATURE_CROSS 句の適用

BigQuery ML には、構文 `ML.FEATURE_CROSS(STRUCT(features), degree)` で特徴量の交差を実行する前処理句 `ML.FEATURE_CROSS` が用意されています。ここで、features はカンマ区切りのカテゴリカル列であり、degree はすべての組み合わせの最大次数です。

#### 特徴量交差を含むモデルの作成

In [ ]:
%%bigquery
CREATE OR REPLACE MODEL
    babyweight.model_1

OPTIONS (
    MODEL_TYPE="LINEAR_REG",
    INPUT_LABEL_COLS=["weight_pounds"],
    L2_REG=0.1,
    DATA_SPLIT_METHOD="NO_SPLIT") AS

SELECT
    # TODO: 基本特徴量とラベルを追加
    ML.FEATURE_CROSS(
        # TODO: カテゴリカル特徴量を交差
    ) AS gender_plurality_cross
FROM
    babyweight.babyweight_data_train

#### モデルを評価する 2 つの SQL ステートメントの作成

In [ ]:
%%bigquery
SELECT
    *
FROM
    ML.EVALUATE(MODEL babyweight.model_1,
    (
    SELECT
        # TODO: トレーニング時と同じ特徴量とラベルを追加
    FROM
        babyweight.babyweight_data_eval
    ))

In [ ]:
%%bigquery
SELECT
    # TODO: 計算された RMSE のみを選択
FROM
    ML.EVALUATE(MODEL babyweight.model_1,
    (
    SELECT
        # TODO: トレーニング時と同じ特徴量とラベルを追加
    FROM
        babyweight.babyweight_data_eval
    ))

## ラボ課題 2: モデル 2: BUCKETIZE 関数の適用
 
Bucketize は「バケット」（ビンなど）を作成する前処理関数です。たとえば、構文 `ML.BUCKETIZE(feature, split_points)` を使用して、連続値の数値特徴量をバケット名を値とする文字列特徴量にバケット化します。ここで、split_points はバケットの境界を決定する数値ポイントの配列です。

#### FEATURE_CROSS 内での BUCKETIZE 関数の適用
* ヒント: model_2 を作成します。

In [ ]:
%%bigquery
CREATE OR REPLACE MODEL
    babyweight.model_2

OPTIONS (
    MODEL_TYPE="LINEAR_REG",
    INPUT_LABEL_COLS=["weight_pounds"],
    L2_REG=0.1,
    DATA_SPLIT_METHOD="NO_SPLIT") AS

SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks,
    ML.FEATURE_CROSS(
        STRUCT(
            is_male,
            ML.BUCKETIZE(
                # TODO: mother_age をバケット化
            ) AS bucketed_mothers_age,
            plurality,
            ML.BUCKETIZE(
                # TODO: gestation_weeks をバケット化
            ) AS bucketed_gestation_weeks
        )
    ) AS crossed
FROM
    babyweight.babyweight_data_train

#### モデルを評価（EVALUATE）する 3 つの SQL ステートメントの作成

トレーニングの統計情報を取得し、モデルを評価しましょう。

In [ ]:
%%bigquery
SELECT * FROM ML.TRAINING_INFO(MODEL babyweight.model_2)

それでは、eval データセットでモデルを評価します。

In [ ]:
%%bigquery
SELECT
    *
FROM
    ML.EVALUATE(MODEL babyweight.model_2,
    (
    SELECT
        # TODO: トレーニング時と同じ特徴量とラベルを追加
    FROM
        babyweight.babyweight_data_eval))

先ほど計算した評価テーブルから `mean_squared_error` を選択し、その平方根（SQRT）を取って RMSE を取得しましょう。

In [ ]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL babyweight.model_2,
    (
    SELECT
        # TODO: トレーニング時と同じ特徴量とラベルを追加
    FROM
        babyweight.babyweight_data_eval))

## ラボ課題 3: モデル 3: TRANSFORM 句の適用

予測を実行する前に、特徴量セット全体を TRANSFORM 句にカプセル化する必要があります。これにより、クエリを変更することなく、トレーニングと予測に同じ変換を適用できます。

model_3 に TRANSFORM 句を適用してクエリを実行しましょう。

In [ ]:
%%bigquery
CREATE OR REPLACE MODEL
    babyweight.model_3

TRANSFORM(
    # TODO: SELECT で行うのと同様に基本特徴量とラベルを追加
    # TODO: SELECT で行うのと同様に変換された特徴量を追加
)

OPTIONS (
    MODEL_TYPE="LINEAR_REG",
    INPUT_LABEL_COLS=["weight_pounds"],
    L2_REG=0.1,
    DATA_SPLIT_METHOD="NO_SPLIT") AS

SELECT
    *
FROM
    babyweight.babyweight_data_train

トレーニングの統計情報を取得しましょう。

In [ ]:
%%bigquery
SELECT * FROM ML.TRAINING_INFO(MODEL babyweight.model_3)

それでは、eval データセットでモデルを評価します。

In [ ]:
%%bigquery
SELECT
    *
FROM
    ML.EVALUATE(MODEL babyweight.model_3,
    (
    SELECT
        *
    FROM
        babyweight.babyweight_data_eval
    ))

先ほど計算した評価テーブルから `mean_squared_error` を選択し、その平方根（SQRT）を取って RMSE を取得しましょう。

In [ ]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL babyweight.model_3,
    (
    SELECT
        *
    FROM
        babyweight.babyweight_data_eval
    ))

## ラボのまとめ:
このラボでは、BigQuery の `ML.FEATURE_CROSS` を使用した線形モデルの作成と評価、BigQuery の `ML.FEATURE_CROSS` および `ML.BUCKETIZE` を使用した線形モデルの作成と評価、そして BigQuery の `ML.TRANSFORM` と L2 正則化を使用した線形モデルの作成と評価を行いました。

Copyright 2020 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.